In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. LOAD DATA
# ============================================================
csv_file_path = r'E:\academics\Thesis work\DATAS\grounddata_COMBINED_2Copy_edited.csv'
df = pd.read_csv(csv_file_path)

# Display basic info
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:\n{df.head()}")

In [ ]:
# ============================================================
# 2. PREPARE FEATURES AND TARGET
# ============================================================
target_col = 'Total Biomass of a plot (kg)'
drop_cols = ['Total Biomass of a plot (kg)', 'CMVI', 'EVI', 'MI']

X = df.drop(drop_cols, axis=1)
y = df[target_col]

# Handle non-numeric columns (if any)
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    print(f"\n Dropping non-numeric columns: {non_numeric_cols}")
    X = X.drop(non_numeric_cols, axis=1)

# Drop any remaining columns with NaN values
X = X.dropna(axis=1)
print(f"\nFinal features ({len(X.columns)}): {list(X.columns)}")


In [ ]:
# ============================================================
# 3. TRAIN-TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
# ============================================================
# 4. STEPWISE REGRESSION FUNCTION
# ============================================================
def stepwise_selection(X, y, initial_list=[], threshold_in=0.01, threshold_out=0.05, verbose=True):
    """
    Perform forward-backward stepwise regression.
    """
    included = list(initial_list)
    while True:
        changed = False
        
        # Forward step: try adding each excluded feature
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print(f'  ➕ Add: {best_feature}, p-value: {best_pval:.4f}')
        
        # Backward step: try removing each included feature
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        pvalues = model.pvalues.iloc[1:]  # exclude intercept
        worst_pval = pvalues.max()
        
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            changed = True
            if verbose:
                print(f'  ➖ Remove: {worst_feature}, p-value: {worst_pval:.4f}')
        
        if not changed:
            break
    
    return included

In [ ]:
# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================
def evaluate_model(y_true, y_pred, model_name):
    """Calculate and print regression metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{'─' * 50}")
    print(f" {model_name} RESULTS")
    print(f"{'─' * 50}")
    print(f"  R² Score:     {r2:.4f}")
    print(f"  RMSE:         {rmse:.4f}")
    print(f"  MAE:          {mae:.4f}")
    print(f"  MSE:          {mse:.4f}")
    
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MSE': mse}

In [ ]:
# ============================================================
# 6. MODEL 1: STEPWISE REGRESSION
# ============================================================
print("\n" + "=" * 60)
print("MODEL 1: STEPWISE REGRESSION")
print("=" * 60)

selected_features = stepwise_selection(X_train, y_train, verbose=True)
print(f"\n✅ Selected features ({len(selected_features)}): {selected_features}")

# Fit final stepwise model
X_train_step = sm.add_constant(X_train[selected_features])
X_test_step = sm.add_constant(X_test[selected_features])

model_stepwise = sm.OLS(y_train, X_train_step).fit()
print(f"\n{model_stepwise.summary()}")

# Predictions
y_pred_step_train = model_stepwise.predict(X_train_step)
y_pred_step_test = model_stepwise.predict(X_test_step)

metrics_stepwise = evaluate_model(y_test, y_pred_step_test, "STEPWISE REGRESSION (Test)")

In [ ]:
# ============================================================
# 7. MODEL 2: MULTIPLE LINEAR REGRESSION (ALL FEATURES)
# ============================================================
print("\n" + "=" * 60)
print("MODEL 2: MULTIPLE LINEAR REGRESSION (ALL FEATURES)")
print("=" * 60)

X_train_multi = sm.add_constant(X_train)
X_test_multi = sm.add_constant(X_test)

model_multi = sm.OLS(y_train, X_train_multi).fit()
print(f"\n{model_multi.summary()}")

# Predictions
y_pred_multi_train = model_multi.predict(X_train_multi)
y_pred_multi_test = model_multi.predict(X_test_multi)

metrics_multi = evaluate_model(y_test, y_pred_multi_test, "MULTIPLE LINEAR REGRESSION (Test)")

In [ ]:
# ============================================================
# 8. MODEL 3: SIMPLE LINEAR REGRESSION (SKLEARN)
# ============================================================
print("\n" + "=" * 60)
print("MODEL 3: LINEAR REGRESSION (SKLEARN - ALL FEATURES)")
print("=" * 60)

model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# Predictions
y_pred_sklearn_train = model_sklearn.predict(X_train)
y_pred_sklearn_test = model_sklearn.predict(X_test)

metrics_sklearn = evaluate_model(y_test, y_pred_sklearn_test, "SKLEARN LINEAR REGRESSION (Test)")

# Feature importance (coefficients)
print(f"\n  Coefficients:")
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model_sklearn.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))
print(f"\n  Intercept: {model_sklearn.intercept_:.4f}")

In [ ]:
# ============================================================
# 9. COMPARISON TABLE
# ============================================================
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Stepwise Regression': metrics_stepwise,
    'Multiple Linear Regression (All Features)': metrics_multi,
    'Sklearn Linear Regression': metrics_sklearn
}).T

print(f"\n{comparison.round(4).to_string()}")

In [ ]:
# ============================================================
# 10. VISUALIZATION: PREDICTED vs ACTUAL
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_data = [
    (y_pred_step_test, "Stepwise Regression", metrics_stepwise['R2']),
    (y_pred_multi_test, "Multiple Linear Regression", metrics_multi['R2']),
    (y_pred_sklearn_test, "Sklearn Linear Regression", metrics_sklearn['R2'])
]

for ax, (y_pred, title, r2) in zip(axes, models_data):
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual Biomass (kg)', fontsize=11)
    ax.set_ylabel('Predicted Biomass (kg)', fontsize=11)
    ax.set_title(f'{title}\nR² = {r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 11. RESIDUAL ANALYSIS (for best model)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use stepwise model for residual analysis
residuals = y_test - y_pred_step_test

# Residuals vs Predicted
axes[0].scatter(y_pred_step_test, residuals, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Biomass (kg)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted (Stepwise Model)')
axes[0].grid(True, alpha=0.3)

# Q-Q plot for normality
sm.qqplot(residuals, line='45', fit=True, ax=axes[1])
axes[1].set_title('Q-Q Plot (Stepwise Model)')

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 12. SAVE RESULTS
# ============================================================
# Save predictions to CSV
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Stepwise_Predicted': y_pred_step_test.values,
    'MultipleLR_Predicted': y_pred_multi_test.values,
    'Sklearn_Predicted': y_pred_sklearn_test
})
results_df.to_csv('regression_predictions_comparison.csv', index=False)
print("\n Predictions saved to: regression_predictions_comparison.csv")

# Save model comparison
comparison.to_csv('model_comparison_metrics.csv')
print("Comparison metrics saved to: model_comparison_metrics.csv")

print("\n" + "=" * 60)
print(" ALL MODELS COMPLETED SUCCESSFULLY!")
print("=" * 60)